# Cohort B: assign each cancer episode to a care site

Cohort B episodes come purely from the EHR (no registry confirmation), so this notebook adds a
data-quality filter: for each episode, look at where the patient's cancer-coded visits actually
happened in the surrounding window, and keep only episodes where a meaningful share of that care
happened at a small set of known UNC-affiliated locations (Rockingham, McCreary, Hillsboro, "main",
or Rex). This filters out episodes that are mostly incidental EHR mentions of a cancer diagnosis
made/treated elsewhere.

Output: `cohort_b_derived.cohort_b_care_sites`, used by `06_set_up_cohort_b_feature_table`.

In [ ]:
%sql
USE CATALOG your_catalog;

### Build per-visit care-site utilization
For each Cohort B episode, looks at visits with a cancer ("C") diagnosis code in a window from 90
days before to 180 days after the episode's index date, maps each visit to a care-site location,
buckets known `location_id`s into named site categories, and computes what percentage of a
patient's visits (per episode) fell into each site category.

In [ ]:
%sql
create or replace temp view utilization as (
WITH pre as (
  select ehr_person_id, ehr_rollup2, ehr_episode_start as index_date
  from cohort_b_derived.cohort_b_source
  ),


cancers as (
select p.ehr_person_id, p.ehr_rollup2, p.index_date, v.visit_occurrence_id, v.visit_start_date
from pre p
left join cohort_b.visit_occurrence v on p.ehr_person_id = v.person_id 
JOIN cohort_b.condition_occurrence co ON v.visit_occurrence_id = co.visit_occurrence_id
JOIN all_c_codes_in_data a ON co.condition_concept_id = a.original_concept_id
),

visits as (
select o.ehr_person_id, o.index_date, o.ehr_rollup2, o.visit_occurrence_id, vd.care_site_id, cs.care_site_name
from cancers o 
left join cohort_b.visit_detail vd
on o.ehr_person_id = vd.person_id and vd.visit_occurrence_id = o.visit_occurrence_id
left join cohort_a.care_site cs
on cs.care_site_id = vd.care_site_id
where o.visit_start_date between date_sub(o.index_date, 90) and date_add(o.index_date,180)),

sites as 
(select t.ehr_person_id, t.ehr_rollup2, t.index_date, t.visit_occurrence_id, t.care_site_id, t.care_site_name, l.location_id,
case when l.location_id in (23681) then 'Rockingham'
when l.location_id in (31885,23657) then 'McCreary'
when l.location_id in (36214) then 'Hillsboro'
when l.location_id in (14562) and 
(lower(l.care_site_name) like '%uncmh%' 
OR lower(l.care_site_name)  like '%uncw%' 
OR lower(l.care_site_name) like '%uncnh%' 
OR lower(l.care_site_name)  like '%uncca%'
OR lower(l.care_site_name) like '%cancer hosp%'
OR lower(l.care_site_name) like '%oncology%'
OR lower(l.care_site_name) like '%cancer support%') then 'main'
when l.location_id in (62130, 62085,62043,62031,56887,61910, 62107,36855,29911,28632,13147,3782,
13107,
13156,
14562,
14714,
17013,
17066,
31101,
40355,
41368,
61967,
62125,
62140,
62164,
70907,
70932,
77512,
82590) then 'rex'
else 'nonreportable' end as care_site_cat
from visits t
left join cohort_a.care_site l
on l.care_site_id = t.care_site_id
),

util as(
select ehr_person_id, ehr_rollup2, index_date,  care_site_cat, count(*) *100 /sum(count(*)) OVER (partition by ehr_person_id, ehr_rollup2)  as percent
from sites 
group by ehr_person_id, ehr_rollup2, index_date, care_site_cat)


select * from util
 )

Load the utilization view into a Spark DataFrame.

In [ ]:
utilization_df = spark.sql("""
SELECT * FROM utilization""")

Pivot to one row per patient/episode, with one column per care-site category (% of visits).

In [ ]:
import pyspark.sql.functions as F
pivot_utilization = (
    utilization_df
    .groupBy("ehr_person_id", "ehr_rollup2", "index_date")
    .pivot("care_site_cat")
    .agg(F.max("percent"))
    .fillna(0)
)

Register the pivoted table as a temp view for the final SQL filter.

In [ ]:
pivot_utilization.createOrReplaceTempView("pivot_utilization")

### Apply the care-site threshold
Keeps episodes where at least 25% of nearby visits happened at Hillsboro, McCreary, Rockingham, or
the "main" UNC cancer hospital sites.

In [ ]:
%sql 
create or replace table cohort_b_derived.cohort_b_care_sites as
select * 
from pivot_utilization
where  (Hillsboro >25 OR McCreary >25 OR Rockingham >25 OR main >25)